# Seguridad de la API GraphQL

SAIF aplicado al servicio real `tp_mlops2.graphql_api`, que corre en
`http://localhost:8001/graphql`. Los controles -introspeccion deshabilitada, limite de
profundidad de consulta y autorizacion por campo- viven en
`src/tp_mlops2/graphql_api/schema.py` y `main.py`, activos en el contenedor
`graphql-api` todo el tiempo. Este notebook los prueba contra el servicio real por HTTP.

In [1]:
import requests

BASE_URL = "http://localhost:8001/graphql"


def query(q, token=None):
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    r = requests.post(BASE_URL, json={"query": q}, headers=headers, timeout=10)
    return r.json()

## 1. Introspeccion: deshabilitada

Sin este control, cualquiera podria explorar la estructura completa del esquema
(`__schema`) y mapear toda la API sin tener acceso a ningun dato.

In [2]:
r = query("{ __schema { types { name } } }")
print("introspeccion bloqueada:", "errors" in r)
print(r["errors"][0]["message"])

introspeccion bloqueada: True
GraphQL introspection has been disabled, but the requested query contained the field '__schema'.


## 2. `model`: publico, sin autorizacion por campo

Las metricas del modelo (MAE, MAPE) se pueden consultar sin JWT, igual que
`/v1/model/info` en REST.

In [3]:
r = query('{ model(name: "random_forest_demanda") { name version metrics { maeMw mape } } }')
print(r)

{'data': {'model': {'name': 'random_forest_demanda', 'version': '8', 'metrics': {'maeMw': 1736.815230933058, 'mape': 0.05999729717375087}}}}


## 3. `lineage`: requiere rol `analista` o `admin`

El linaje completo viaja desde una consulta a Neo4j, mas costosa y mas sensible que las
metricas. El campo se autoriza por separado, leyendo el rol del JWT en el contexto de la
request.

In [4]:
r_sin_rol = query("{ lineage { name kind } }")
print("sin token:", r_sin_rol["errors"][0]["message"])

sin token: no autorizado: el campo lineage requiere rol analista o admin


In [5]:
from tp_mlops2.auth import crear_token

token_analista = crear_token("alan", "analista")
r_con_rol = query("{ lineage { name kind } }", token=token_analista)
nodos = r_con_rol["data"]["lineage"]
print(f"con rol analista: {len(nodos)} nodos reales desde Neo4j")

con rol analista: 18 nodos reales desde Neo4j


## 4. Limite de profundidad de consulta

El esquema real limita la profundidad a 4 niveles (`QueryDepthLimiter(max_depth=4)` en
`schema.py`). Una consulta razonable como `model { metrics { maeMw } }` tiene profundidad
3 y pasa sin problema.

In [6]:
query_normal = '{ model(name: "random_forest_demanda") { name metrics { maeMw } } }'
r = query(query_normal)
print("profundidad 3, dentro del limite de 4:", "errors" not in r, r.get("data"))

profundidad 3, dentro del limite de 4: True {'model': {'name': 'random_forest_demanda', 'metrics': {'maeMw': 1736.815230933058}}}


## Reflexion

Los tres controles -sin introspeccion, limite de profundidad, y autorizacion por campo en
`lineage`- estan activos en el contenedor `graphql-api` en todo momento, no solo en esta
demostracion. El JWT que autoriza `lineage` es el mismo mecanismo que protege
`/v1/model/metrics` en REST (`tp_mlops2.auth`), asi que un token emitido para un rol vale
en las dos capas sin duplicar logica de autenticacion.